# Support Vector Machines (SVM), Explained Simply
The widest street between two classes, and the kernel trick for curved boundaries. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A bottle factory measures each bottle's wall thickness (mm) and weight (g), and a worker marks it OK (1) or faulty (0).

### 120 bottles
- `bottles.csv`: 120 bottles: wall thickness in mm, weight in grams, ok (1) or faulty (0)

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "support-vector-machines/bottles.csv")
bottles = pd.read_csv(url)
print(bottles.shape)
print(bottles["ok"].value_counts().to_dict())
bottles.head()

## 2. A linear SVM
An SVM finds the line with the **widest margin**: the widest empty street between the two classes. The points on the edges of the street are the **support vectors**.

### A helper to draw the street
- `decision_function`: how far each point is from the middle line (0), measured in street half-widths
- `levels=[-1, 0, 1]`: the middle line (solid) and the two edges of the street (dashed)
- `contourf(... zz > 0 ...)`: shade the OK side green and the faulty side red

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def draw(model, X, y, ax, title):
    a, b = X.iloc[:, 0], X.iloc[:, 1]
    xx, yy = np.meshgrid(np.linspace(a.min() - 0.3, a.max() + 0.3, 300),
                         np.linspace(b.min() - 3, b.max() + 3, 300))
    grid = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=X.columns)
    zz = model.decision_function(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz > 0, alpha=0.2, cmap="RdYlGn")
    ax.contour(xx, yy, zz, levels=[-1, 0, 1], colors="k",
               linestyles=["--", "-", "--"])
    ax.scatter(X.iloc[:, 0], X.iloc[:, 1], c=y, cmap="RdYlGn", edgecolors="k", s=25)
    ax.set_title(title)

### Fit a linear SVM
- `SVC(kernel="linear")`: a support vector machine with a straight boundary
- `StandardScaler()`: SVMs measure distances, so scale first
- `n_support_`: how many support vectors on each side: only these points decide the line

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = bottles[["wall_mm", "weight_g"]]
y = bottles["ok"]
svm = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1))
svm.fit(X, y)
print("support vectors per class:", svm[-1].n_support_)
fig, ax = plt.subplots(figsize=(6, 5))
draw(svm, X, y, ax, "linear SVM, C = 1"); plt.show()

### Small C vs big C
- `C = 0.01`: a soft margin: a wide street that lets some points inside
- `C = 100`: a hard margin: a narrow street that tries never to let a point in
- `the titles`: many support vectors for small C, very few for big C

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, C in zip(axes, [0.01, 100]):
    m = make_pipeline(StandardScaler(), SVC(kernel="linear", C=C))
    m.fit(X, y)
    draw(m, X, y, ax, f"C = {C}  ({m[-1].n_support_.sum()} support vectors)")
plt.show()

## 3. When no straight line works
An engine is safe when temperature and pressure are both near their normal values: safe readings sit in the middle, unsafe ones all around them. The **kernel trick** lets an SVM draw curved boundaries.

### Engine readings
- `url.replace(...)`: the second file sits in the same folder
- `safe`: 1 = safe, 0 = unsafe

In [ ]:
url2 = url.replace("bottles.csv", "engine_readings.csv")
engine = pd.read_csv(url2)
Xe = engine[["temperature_c", "pressure_bar"]]
ye = engine["safe"]
print(engine.shape, ye.value_counts().to_dict())

### Linear vs RBF kernel
- `kernel="linear"`: a straight line: hopeless for a ring
- `kernel="rbf"`: the radial basis function kernel: boundaries that can curve around

In [ ]:
from sklearn.model_selection import train_test_split

Xe_tr, Xe_te, ye_tr, ye_te = train_test_split(
    Xe, ye, test_size=0.3, random_state=0)
for k in ["linear", "rbf"]:
    m = make_pipeline(StandardScaler(), SVC(kernel=k)).fit(Xe_tr, ye_tr)
    print(f"{k:>6} kernel: test accuracy {m.score(Xe_te, ye_te):.2f}")

### What gamma does
- `gamma`: how far each training point's influence reaches
- `gamma = 0.01`: far-reaching: too smooth, it can't bend around the safe zone (underfitting)
- `gamma = 50`: very local: blobs around single points, overfitting
- `gamma = 1`: about right

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, g in zip(axes, [0.01, 1, 50]):
    m = make_pipeline(StandardScaler(), SVC(kernel="rbf", gamma=g))
    m.fit(Xe_tr, ye_tr)
    draw(m, Xe, ye, ax, f"gamma = {g}  (test {m.score(Xe_te, ye_te):.2f})")
plt.show()

## 4. Tuning C and gamma
C and gamma work together, so search them together with cross-validation.

### Grid search
- `svc__C`: the C setting of the step named svc in the pipeline
- `cv=5`: 5-fold cross-validation on the training readings
- `search.score`: accuracy of the best model on the test readings

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.1, 0.5, 1, 5]}
search = GridSearchCV(pipe, grid, cv=5).fit(Xe_tr, ye_tr)
print("best:", search.best_params_)
print("test accuracy:", round(search.score(Xe_te, ye_te), 3))

## Try it yourself
1. Check a bottle with a 2.2 mm wall weighing 395 g.
2. On the engine data, try gamma=0.1 and gamma=100. What happens to the boundary?
3. Try kernel='poly' with degree=2 on the engine data.